In [1]:
# 7-7-2026

In [2]:
import pandas as pd
import numpy as np
from scipy.spatial import distance_matrix
from scipy.stats import spearmanr, kendalltau

In [3]:
# load embeddings, set domain_id as index
df_embeddings = pd.read_csv("domain_embeddings_v2.csv")
df_embeddings.set_index("domain_id", inplace=True)

In [4]:
df_embeddings.head()

,e_0,e_1,e_2,e_3,e_4,e_5,e_6,e_7
domain_id,,,,,,,,
0,0.320788,0.029515,0.493621,-0.254550,0.157485,0.043007,0.223810,-0.180119
1,0.035994,0.512844,0.735541,-1.007554,0.671355,0.219516,0.482659,-0.768034
2,0.258181,-0.175929,0.318462,0.016263,-0.086862,0.120247,-0.007532,-0.090168
4,0.360256,-0.026251,0.126985,0.132024,0.164264,-0.143251,-0.185669,0.135234
5,0.374883,-0.071358,0.552073,-0.290540,0.058007,0.202109,0.179068,-0.340887


In [5]:
# raw embedding distance matrix, no normalization
dist_raw = distance_matrix(df_embeddings.values, df_embeddings.values) # eucliean by default
# no scaling because the fact that some nodes have greater scales than others actually means something for downstream prediction
# nodes having differnt scales actually means something to the network

C:\Users\Yash\AppData\Local\Temp\ipykernel_33872\2764295889.py:2: DeprecationWarning: `distance_matrix` is deprecated in favor of `scipy.spatial.distance.cdist` as of SciPy 1.18.0 and will be removed in SciPy 1.20.0.
  dist_raw = distance_matrix(df_embeddings.values, df_embeddings.values) # eucliean by default
C:\Users\Yash\AppData\Local\Temp\ipykernel_33872\2764295889.py:2: DeprecationWarning: `minkowski_distance` is deprecated in favor of `scipy.spatial.distance.minkowski` as of SciPy 1.18.0 and will be removed in SciPy 1.20.0.
  dist_raw = distance_matrix(df_embeddings.values, df_embeddings.values) # eucliean by default
C:\Users\Yash\AppData\Local\Temp\ipykernel_33872\2764295889.py:2: DeprecationWarning: `minkowski_distance_p` is deprecated in favor of `scipy.spatial.distance.minkowski` as of SciPy 1.18.0 and will be removed in SciPy 1.20.0.
  dist_raw = distance_matrix(df_embeddings.values, df_embeddings.values) # eucliean by default


In [6]:
# negate so bigger=lower transferability. same  as rawdist
dist_raw_df = pd.DataFrame(-dist_raw, index=df_embeddings.index, columns=df_embeddings.index)

In [7]:
dist_raw_df.head()

domain_id,0,1,2,4,5,6,7,8,11,12,...,32,33,36,37,38,39,45,46,47,49
domain_id,,,,,,,,,,,,,,,,,,,,,
0,-0.000000,-1.283791,-0.526716,-0.768367,-0.284379,-0.730119,-0.813101,-0.586308,-0.539763,-1.386452,...,-0.883986,-0.553056,-0.692510,-0.909542,-0.431134,-2.958495,-1.050950,-0.930292,-0.609290,-0.562334
1,-1.283791,-0.000000,-1.740825,-1.927705,-1.286479,-1.955912,-2.036067,-0.763700,-1.584322,-1.953257,...,-1.990409,-1.747790,-1.944240,-1.310502,-1.659987,-3.069270,-1.923802,-2.143102,-1.788747,-1.411729
2,-0.526716,-1.740825,-0.000000,-0.545822,-0.546466,-0.458187,-0.587568,-1.026400,-0.548009,-1.517400,...,-0.855306,-0.517830,-0.450876,-1.178803,-0.347590,-3.192266,-0.920493,-0.775952,-0.369576,-0.675117
4,-0.768367,-1.927705,-0.545822,-0.000000,-0.922929,-0.251164,-0.456162,-1.211518,-0.973311,-1.162037,...,-0.822570,-0.440507,-0.382822,-1.592120,-0.486005,-2.840285,-1.393789,-0.819681,-0.221860,-0.615110
5,-0.284379,-1.286479,-0.546466,-0.922929,-0.000000,-0.881718,-0.957936,-0.653677,-0.346508,-1.597839,...,-0.951680,-0.737359,-0.835023,-0.709408,-0.601795,-3.152650,-0.819343,-0.998892,-0.748704,-0.763223


In [8]:
dist_raw_df.to_csv("embedding_matrix_v2.csv")

In [9]:
embedding_matrix = pd.read_csv("embedding_matrix_v2.csv")
embedding_matrix.set_index("domain_id", inplace=True)
embedding_matrix.index.name = None
embedding_matrix.index = embedding_matrix.index.astype(int)
embedding_matrix.columns = embedding_matrix.columns.astype(int)
embedding_matrix.head(10)

,0,1,2,4,5,6,7,8,11,12,...,32,33,36,37,38,39,45,46,47,49
0,-0.000000,-1.283791,-0.526716,-0.768367,-0.284379,-0.730119,-0.813101,-0.586308,-0.539763,-1.386452,...,-0.883986,-0.553056,-0.692510,-0.909542,-0.431134,-2.958495,-1.050950,-0.930292,-0.609290,-0.562334
1,-1.283791,-0.000000,-1.740825,-1.927705,-1.286479,-1.955912,-2.036067,-0.763700,-1.584322,-1.953257,...,-1.990409,-1.747790,-1.944240,-1.310502,-1.659987,-3.069270,-1.923802,-2.143102,-1.788747,-1.411729
2,-0.526716,-1.740825,-0.000000,-0.545822,-0.546466,-0.458187,-0.587568,-1.026400,-0.548009,-1.517400,...,-0.855306,-0.517830,-0.450876,-1.178803,-0.347590,-3.192266,-0.920493,-0.775952,-0.369576,-0.675117
4,-0.768367,-1.927705,-0.545822,-0.000000,-0.922929,-0.251164,-0.456162,-1.211518,-0.973311,-1.162037,...,-0.822570,-0.440507,-0.382822,-1.592120,-0.486005,-2.840285,-1.393789,-0.819681,-0.221860,-0.615110
5,-0.284379,-1.286479,-0.546466,-0.922929,-0.000000,-0.881718,-0.957936,-0.653677,-0.346508,-1.597839,...,-0.951680,-0.737359,-0.835023,-0.709408,-0.601795,-3.152650,-0.819343,-0.998892,-0.748704,-0.763223
6,-0.730119,-1.955912,-0.458187,-0.251164,-0.881718,-0.000000,-0.257667,-1.230507,-0.905272,-1.307041,...,-0.730740,-0.318748,-0.158890,-1.518412,-0.361258,-2.961634,-1.299632,-0.642675,-0.254603,-0.683727
7,-0.813101,-2.036067,-0.587568,-0.456162,-0.957936,-0.257667,-0.000000,-1.318376,-0.952568,-1.403390,...,-0.650851,-0.310348,-0.182516,-1.532465,-0.427538,-3.000309,-1.322912,-0.519273,-0.476248,-0.835633
8,-0.586308,-0.763700,-1.026400,-1.211518,-0.653677,-1.230507,-1.318376,-0.000000,-0.968850,-1.515496,...,-1.394073,-1.046219,-1.221534,-1.001201,-0.939273,-2.974545,-1.425049,-1.490644,-1.076751,-0.725305
11,-0.539763,-1.584322,-0.548009,-0.973311,-0.346508,-0.905272,-0.952568,-0.968850,-0.000000,-1.770127,...,-0.912364,-0.814170,-0.831876,-0.694319,-0.694216,-3.329887,-0.560040,-0.893432,-0.824852,-1.003250
12,-1.386452,-1.953257,-1.517400,-1.162037,-1.597839,-1.307041,-1.403390,-1.515496,-1.770127,-0.000000,...,-1.559536,-1.295563,-1.389507,-2.169511,-1.315835,-1.759209,-2.201983,-1.681283,-1.218390,-1.102762


In [10]:
rawdist_matrix = pd.read_csv("../baselines/rawdist_matrix.csv")
rawdist_matrix.set_index("Unnamed: 0", inplace=True)
rawdist_matrix.index.name = None
rawdist_matrix.index = rawdist_matrix.index.astype(int)
rawdist_matrix.columns = rawdist_matrix.columns.astype(int)
rawdist_matrix.head(10)

,0,1,2,4,5,6,7,8,11,12,...,32,33,36,37,38,39,45,46,47,49
0,-0.000000,-6.672423,-11.045133,-7.986460,-3.308284,-7.958821,-6.068376,-4.666705,-5.184888,-4.589527,...,-7.662378,-6.407206,-8.197987,-5.296501,-4.204545,-11.505974,-7.273316,-13.276326,-6.751253,-3.125536
1,-6.672423,-0.000000,-13.599319,-9.841238,-5.547623,-9.436429,-9.730264,-5.698760,-8.974480,-8.392720,...,-11.038897,-10.159206,-10.744922,-5.711799,-7.998925,-13.061897,-11.181847,-15.567050,-8.537365,-6.931799
2,-11.045133,-13.599319,-0.000000,-13.082436,-11.499412,-12.708249,-9.285989,-13.684210,-9.565427,-8.687225,...,-7.894825,-9.563264,-10.696453,-11.590126,-9.063832,-9.620238,-7.577379,-16.658596,-9.353973,-11.288658
4,-7.986460,-9.841238,-13.082436,-0.000000,-8.019986,-3.351193,-7.878595,-9.379059,-9.467587,-9.484902,...,-10.290864,-7.254905,-6.459036,-9.505884,-7.903673,-13.860378,-11.642435,-9.120748,-5.452087,-7.889157
5,-3.308284,-5.547623,-11.499412,-8.019986,-0.000000,-7.551036,-6.591061,-4.260023,-4.860621,-5.079144,...,-8.673302,-7.712744,-8.004912,-4.035172,-5.374218,-12.376469,-7.461498,-12.675088,-6.924344,-4.980700
6,-7.958821,-9.436429,-12.708249,-3.351193,-7.551036,-0.000000,-7.307292,-8.895403,-9.258151,-9.329919,...,-9.975772,-7.488843,-4.461359,-9.152761,-7.552179,-13.639737,-11.240952,-8.347249,-5.771723,-7.863910
7,-6.068376,-9.730264,-9.285989,-7.878595,-6.591061,-7.307292,-0.000000,-8.832793,-7.072502,-5.136909,...,-4.850997,-4.276811,-5.501850,-8.444666,-2.986468,-8.724586,-7.209533,-12.837734,-5.064828,-5.701705
8,-4.666705,-5.698760,-13.684210,-9.379059,-4.260023,-8.895403,-8.832793,-0.000000,-7.504786,-7.592999,...,-10.594288,-8.880334,-9.930977,-5.531485,-7.389167,-14.100157,-10.431665,-13.559093,-9.241698,-5.032279
11,-5.184888,-8.974480,-9.565427,-9.467587,-4.860621,-9.258151,-7.072502,-7.504786,-0.000000,-5.607953,...,-8.563762,-8.084519,-8.778158,-4.434500,-6.187654,-12.536701,-4.309561,-13.146039,-7.716434,-6.559332
12,-4.589527,-8.392720,-8.687225,-9.484902,-5.079144,-9.329919,-5.136909,-7.592999,-5.607953,-0.000000,...,-6.457259,-6.402053,-8.537121,-6.857717,-4.288140,-9.152598,-5.360677,-14.267618,-6.997054,-5.922538


In [11]:
true_matrix = pd.read_csv("../transfer-matrix/rf_transfer_matrix.csv")
true_matrix.set_index("Unnamed: 0", inplace=True)
true_matrix.index.name = None
true_matrix.index = true_matrix.index.astype(int)
true_matrix.columns = true_matrix.columns.astype(int)
true_matrix.head(10)

,0,1,2,4,5,6,7,8,11,12,...,32,33,36,37,38,39,45,46,47,49
0,0.392727,0.291660,0.127238,0.058840,0.214817,0.084705,0.149056,0.182358,0.173877,0.026375,...,0.228505,0.243433,0.149609,0.275203,0.112525,-0.011557,0.116862,0.158796,0.025454,0.029928
1,0.142172,0.395760,0.030643,0.010376,0.180707,0.064432,0.056092,0.128808,0.097788,0.039590,...,0.167477,0.032806,0.029704,0.193033,0.055264,0.066550,0.108203,0.199762,0.023240,-0.037710
2,0.196600,0.214445,0.368173,0.023968,0.207323,0.140139,0.172990,0.231019,0.156741,0.057226,...,0.197402,0.223793,0.121457,0.248807,0.134485,0.053503,-0.005610,0.253965,0.031899,0.059566
4,0.235489,0.227173,0.127394,0.410382,0.215954,0.186207,0.125068,0.196603,0.165264,0.034514,...,0.178055,0.245668,0.147579,0.284360,0.217076,-0.003941,0.037643,0.306750,0.137154,0.020968
5,0.190803,0.206530,0.097887,0.087230,0.464604,0.144950,0.159361,0.204884,0.264641,0.104493,...,0.197195,0.232923,0.117228,0.292623,0.170687,-0.020694,0.115293,0.297271,0.067951,0.051770
6,0.178282,0.238235,0.034447,0.220869,0.191487,0.317381,0.118250,0.163631,0.119401,0.021430,...,0.202794,0.247076,0.125801,0.255555,0.201974,-0.068777,0.099243,0.321076,0.102840,-0.017236
7,0.113396,0.054282,0.061550,0.069370,0.044484,0.109329,0.329688,0.092023,0.159883,0.051387,...,0.184071,0.179713,0.142222,0.192045,0.209176,0.035736,0.042035,0.159817,0.008022,0.022467
8,0.181002,0.228547,-0.006209,0.023346,0.133774,0.117255,0.194411,0.424899,0.064289,-0.004919,...,0.093679,0.136592,0.101116,0.173933,0.095452,-0.003312,0.050942,0.240088,0.051530,0.007182
11,0.214508,0.227264,0.109767,0.124790,0.237549,0.076848,0.107019,0.207917,0.551275,-0.003236,...,0.256072,0.166287,0.102690,0.366209,0.181682,-0.007782,0.147501,0.206234,-0.057620,0.064859
12,0.118927,0.175802,-0.016922,0.112255,0.159920,0.070012,0.077276,0.026258,0.103184,0.502334,...,0.163311,-0.091433,0.060118,0.163172,0.119053,0.080449,0.089125,0.118189,0.068853,-0.070408


In [12]:
full_tau, _ = kendalltau(embedding_matrix.values.flatten(), true_matrix.values.flatten())
full_tau

np.float64(0.21443153234281356)

In [13]:
mask = ~np.eye(len(embedding_matrix), dtype=bool)
off_diag_tau, _ = kendalltau(embedding_matrix.values[mask], true_matrix.values[mask])
print(f"off diag kendal tau: {off_diag_tau:.4f}")

off diag kendal tau: 0.1671


In [14]:
full_tau, _ = kendalltau(embedding_matrix.values.flatten(), rawdist_matrix.values.flatten())
full_tau

np.float64(0.4076834896799246)

In [15]:
mask = ~np.eye(len(embedding_matrix), dtype=bool)
off_diag_tau, _ = kendalltau(embedding_matrix.values[mask], rawdist_matrix.values[mask])
print(f"off diag kendal tau: {off_diag_tau:.4f}")

off diag kendal tau: 0.3717


In [16]:
mask = ~np.eye(len(embedding_matrix), dtype=bool)
off_diag_tau, _ = spearmanr(true_matrix.values[mask], embedding_matrix.values[mask])
print(f"off diag sparman: {off_diag_tau:.4f}")

off diag sparman: 0.2464
